**Training with LoRA parameter**

In [ ]:
pip uninstall torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time

In [ ]:
class LoRALayer(nn.Module):
  def __init__(self, model, d, k, r):
    super().__init__()
    self.A = nn.Parameter(torch.randn(r, k))
    self.B = nn.Parameter(torch.zeros(d, r))
    self.model = model
    for param in self.model.parameters():
      param.requires_grad = False

  def forward(self, x):
    xa = x@self.A.T
    xb = xa@ self.B.T
    return self.model(x) + xb

In [ ]:
from transformers import AutoModelForCausalLM

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [ ]:
for params in model.parameters():
  params.requires_grad = False

In [ ]:
for i in range(len(model.transformer.h)):
  x_attn = model.transformer.h[i].attn.c_attn
  model.transformer.h[i].attn.c_attn = LoRALayer(x_attn, d= 2304, k=768, r = 4)
  print(model.transformer.h[i].attn.c_attn)

LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)
LoRALayer(
  (model): Conv1D(nf=2304, nx=768)
)


In [ ]:
total_params = sum(params.numel() for params in model.parameters())
trainable_params = sum(params.numel() for params in model.parameters() if params.requires_grad)
print(f"Total: {total_params}, Trainable: {trainable_params}, Percent: {100 * trainable_params / total_params:.4f}%")

Total: 124587264, Trainable: 147456, Percent: 0.1184%


In [ ]:
from datasets import load_dataset
ds = load_dataset("sh0416/ag_news")

In [ ]:
lr = 1e-3
batch_size = 4
EPOCH = 2

In [ ]:
optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=lr,            # Learning rate (default: 1e-3)
    betas=(0.9, 0.999), # Coefficients for moving averages (default)
    eps=1e-8,           # Numerical stability constant (default)
    weight_decay=1e-2   # True weight decay coefficient (default: 1e-2)
)

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token= tokenizer.eos_token

def tokenize_fn(data):
    return tokenizer(data['description'], truncation=True, padding='max_length', max_length=128)

subset = ds['train'].select(range(5000))
tokenized_data = subset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
tokenized_data.column_names

['label', 'title', 'description', 'input_ids', 'attention_mask']

In [ ]:
tokenized_data.set_format(type = "torch", columns = ['input_ids', 'attention_mask'])

In [ ]:
from torch.utils.data import DataLoader

train_data = DataLoader(tokenized_data, batch_size=batch_size , shuffle = True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): LoRALayer(
            (model): Conv1D(nf=2304, nx=768)
          )
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
total_loss_train_plot = []

start = time.time()
for epoch in range(EPOCH):
  total_loss_train = 0
  for batch in train_data:
    batch={k:v.to(device) for k,v in batch.items()}
    optimizer.zero_grad()
    output = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['input_ids'])
    loss = output.loss
    loss.backward()
    optimizer.step()
    total_loss_train += loss.item()

  total_loss_train_plot.append(round(total_loss_train / len(train_data), 4))
  print(f"Epoch {epoch+1}/{EPOCH}, Train Loss: {round(total_loss_train / len(train_data), 4)}")
  print("==="*25)

end = time.time()
print(f"Total training time: {end-start:.1f}s")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/2, Train Loss: 1.2275
Epoch 2/2, Train Loss: 1.1296
Total training time: 256.9s


**Full Fine-tuning of the model**

In [ ]:
baseline_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
EPOCHS = 2
learning_rate = 5e-5

In [ ]:
ft_optimizer = optim.AdamW(
    baseline_model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=1e-2
)

In [ ]:
ft_total_loss_train_plot = []

start = time.time()
for epoch in range(EPOCHS):
  ft_total_loss_train = 0
  for batch in train_data:
    batch={k:v.to(device) for k,v in batch.items()}
    ft_optimizer.zero_grad()
    output = baseline_model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['input_ids'])
    loss = output.loss
    loss.backward()
    ft_optimizer.step()
    ft_total_loss_train += loss.item()

  ft_total_loss_train_plot.append(round(ft_total_loss_train / len(train_data), 4))
  print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(ft_total_loss_train / len(train_data), 4)}")
  print("==="*25)


end = time.time()
print(f"Total training time: {end-start:.1f}s")

Epoch 1/2, Train Loss: 1.0973
Epoch 2/2, Train Loss: 0.8091
Total training time: 434.4s


In [ ]:
vanilla_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
def perplexity(model, texts, tokenizer, device):
  model.eval()
  total_loss = 0
  count = 0
  with torch.no_grad():
    for text in texts:
      inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=128,
                padding="max_length"
            ).to(device)
      outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                labels=inputs["input_ids"]
            )
      total_loss += outputs.loss.item()
      count += 1
    avg_loss = total_loss / count
  return torch.exp(torch.tensor(total_loss/count))


test_texts = ds["test"].select(range(100))["description"]

baseline_ppl = perplexity(baseline_model, test_texts, tokenizer, device)
lora_ppl = perplexity(model, test_texts, tokenizer, device)
vanilla_ppl = perplexity(vanilla_model, test_texts, tokenizer, device)

print(f"full fine-tuned:      {baseline_ppl:.2f}")
print(f"LoRA fine-tuned:    {lora_ppl:.2f}")
print(f"Vanilla model:       {vanilla_ppl:.2f}")


full fine-tuned:      2.94
LoRA fine-tuned:    3.60
Vanilla model:       4795.95


In [ ]:
import matplotlib.pyplot as plt

# Plot 1: Training loss curves
plt.figure(figsize=(8, 4))
plt.plot(total_loss_train_plot, label="LoRA (0.11% params)", marker='o')
plt.plot(ft_total_loss_train_plot, label="Full fine-tuning (100% params)", marker='s')
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.title("Training Loss: LoRA vs Full Fine-tuning")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

# Plot 2: Perplexity comparison bar chart
plt.figure(figsize=(7, 4))
models = ["Vanilla GPT-2", "LoRA\n(0.11% params, 93.5s)", "Full Fine-tuning\n(100% params, 172.9s)"]
ppls = [4795.95, 3.60, 2.94]
colors = ["#888888", "#4C72B0", "#DD8452"]
bars = plt.bar(models, ppls, color=colors)
plt.ylabel("Perplexity (lower = better)")
plt.title("Perplexity on Held-out AG News Test Set")
plt.yscale("log")  # log scale since vanilla is orders of magnitude higher
for bar, val in zip(bars, ppls):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
             f"{val:.2f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig("perplexity_comparison.png", dpi=150)
plt.show()